# CACE-LOREM 诊断测试

本 notebook 用于在**不改模型结构**前提下，定位 CACE-LOREM 在 cumulene 二面角曲线任务上的瓶颈。

测试内容：
1. 与 LOREM `mp1/mp2` 的曲线误差对比（基于已有 `.npz`）
2. SR/LR 分支贡献统计（`CACE_energy` 与 `lr_energy`）
3. 长程场 `q_field` 通道活性（`l=0/1/2`）

> 说明：该 notebook 只做推理分析，不训练模型。

In [3]:
from pathlib import Path
import numpy as np
import torch

ROOT = Path('/data/home/public/qiuqizhi')
WORKDIR = ROOT / 'LOREM/my_experiments/cumulene/CACE-LOREM'
REPO = ROOT / 'SOG-Qeq/SOG-Net/CACE-LOREM'

np.set_printoptions(precision=6, suppress=True)
print('WORKDIR =', WORKDIR)
print('REPO    =', REPO)
print('torch   =', torch.__version__)

WORKDIR = /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/CACE-LOREM
REPO    = /data/home/public/qiuqizhi/SOG-Qeq/SOG-Net/CACE-LOREM
torch   = 2.6.0+cu124


In [4]:
def curve_metrics(path: Path):
    z = np.load(path)
    y = np.squeeze(z['predicted_energies'])
    t = np.squeeze(z['actual_energies'])
    rmse = float(np.sqrt(np.mean((y - t) ** 2)))
    mae = float(np.mean(np.abs(y - t)))
    corr = float(np.corrcoef(y, t)[0, 1]) if np.std(y) > 0 and np.std(t) > 0 else float('nan')
    amp = float(y.max() - y.min())
    tamp = float(t.max() - t.min())
    return {
        'rmse': rmse,
        'mae': mae,
        'corr': corr,
        'amp': amp,
        'target_amp': tamp,
        'amp_ratio': amp / tamp if tamp > 0 else float('nan'),
    }

paths = {
    'CACE-LOREM': WORKDIR / "CACE_LOREM_MP2_try2" /'cumulene_dihedral_energy_curve.npz',
    'LOREM-mp1': ROOT / 'LOREM/my_experiments/cumulene/lorem-cu35-sr-mp1/cumulene_dihedral_energy_curve.npz',
    'LOREM-mp2': ROOT / 'LOREM/my_experiments/cumulene/lorem-cu35-sr-mp2/cumulene_dihedral_energy_curve.npz',
}

for name, p in paths.items():
    m = curve_metrics(p)
    print(f"{name:10s} rmse={m['rmse']:.6f} mae={m['mae']:.6f} corr={m['corr']:.4f} ",
          f"amp={m['amp']:.6f} amp_ratio={m['amp_ratio']:.4f}")

CACE-LOREM rmse=0.331471 mae=0.246010 corr=-0.2346  amp=0.004329 amp_ratio=0.0058
LOREM-mp1  rmse=0.332710 mae=0.248002 corr=-0.0867  amp=0.000003 amp_ratio=0.0000
LOREM-mp2  rmse=0.015824 mae=0.005352 corr=0.9983  amp=0.684428 amp_ratio=0.9214


In [6]:
import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import cace
from ase import io
try:
    from torch_geometric.dataloader import DataLoader
except ModuleNotFoundError:
    from cace.tools import torch_geometric
    DataLoader = torch_geometric.dataloader.DataLoader
from cace.data.atomic_data import AtomicData

model_path = REPO / 'fit-cumulene/loss_data/CACE_LOREM_MP2_try2/best_model.pth'
# PyTorch 2.6+ 默认 weights_only=True，会拒绝反序列化 CombinePotential。
# 这里模型来自本地训练产物，按可信文件处理。
try:
    model = torch.load(model_path, map_location='cpu', weights_only=False)
except TypeError:
    # 兼容旧版 PyTorch（没有 weights_only 参数）
    model = torch.load(model_path, map_location='cpu')
model.eval()

# 优先使用当前 notebook 环境已有的 systems_profile；否则退化到测试集前 128 个结构
if 'systems_profile' in globals() and len(systems_profile) > 0:
    systems = list(systems_profile)
    print('using existing systems_profile, n =', len(systems))
else:
    systems = io.read(str(ROOT / 'LOREM/datasets/cumulene_test.xyz'), ':128')
    print('systems_profile not found; using cumulene_test subset, n =', len(systems))

cutoff = float(model.models[0].representation.cutoff)
print('cutoff =', cutoff)

systems_profile not found; using cumulene_test subset, n = 128
cutoff = 5.0


In [8]:
def one_output(atoms):
    data = AtomicData.from_atoms(atoms, cutoff=cutoff)
    batch = next(iter(DataLoader([data], batch_size=1)))
    out = model(batch.to_dict(), training=False)
    return out

sr_list, lr_list = [], []
q0_std, q1_std, q2_std = [], [], []

for a in systems:
    out = one_output(a)

    # 分支能量（如果键存在）
    if 'CACE_energy' in out:
        sr_list.append(float(np.squeeze(out['CACE_energy'].detach().cpu().numpy())))
    if 'lr_energy' in out:
        lr_list.append(float(np.squeeze(out['lr_energy'].detach().cpu().numpy())))

    # q_field 活性（如果存在）
    if 'q_field' in out:
        qf = out['q_field'].detach().cpu().numpy()  # [n_atoms, 9]
        q0_std.append(float(np.std(qf[:, :1])))
        q1_std.append(float(np.std(qf[:, 1:4])))
        q2_std.append(float(np.std(qf[:, 4:9])))

print('n_systems =', len(systems))
if sr_list:
    print('SR energy std =', float(np.std(sr_list)), ' range=', float(np.min(sr_list)), float(np.max(sr_list)))
if lr_list:
    print('LR energy std =', float(np.std(lr_list)), ' range=', float(np.min(lr_list)), float(np.max(lr_list)))
    if sr_list:
        denom = float(np.std(sr_list)) + 1e-12
        print('LR/SR std ratio =', float(np.std(lr_list)) / denom)

if q0_std:
    print('q_field std(l0,l1,l2)=', float(np.mean(q0_std)), float(np.mean(q1_std)), float(np.mean(q2_std)))

# 保存诊断结果
np.savez(
    WORKDIR / "CACE_LOREM_MP2_try2"/'diagnostics_summary.npz',
    sr_energy=np.array(sr_list, dtype=np.float64),
    lr_energy=np.array(lr_list, dtype=np.float64),
    q0_std=np.array(q0_std, dtype=np.float64),
    q1_std=np.array(q1_std, dtype=np.float64),
    q2_std=np.array(q2_std, dtype=np.float64),
)
print('saved:', WORKDIR / "CACE_LOREM_MP2_try2"/'diagnostics_summary.npz')

n_systems = 128
SR energy std = 0.5613648788256914  range= -1.2448654174804688 1.5138893127441406
LR energy std = 8.271677174732819e-06  range= -0.5933741331100464 -0.5933327078819275
LR/SR std ratio = 1.4734938872594683e-05
q_field std(l0,l1,l2)= 0.002705213693843689 0.00971023100282764 0.07544363613124005
saved: /data/home/public/qiuqizhi/LOREM/my_experiments/cumulene/CACE-LOREM/CACE_LOREM_MP2_try2/diagnostics_summary.npz
